# Terra — Family Transfer (v2 backbone 비교)

성공한 셀만 추린 클린 버전.  **v3 결과(val 86.0%, family-only 78.8%)와 비교**하려고 백본만 v2 로 바꿔 동일 조건 재학습.

- 데이터셋(`family_v1.npz`)·하이퍼파라미터·정규화 전부 v3 때와 동일 → **백본만 변수**.
- 백본 = `mobilenet_v3_large_v2_best.pth` (100-class, val 86.76%).  로더가 num_classes(100/170) 자동 추론하므로 셀 수정 불필요.
- ONNX 출력명 다르게(`..._v2bb.onnx`) → v3 onnx 안 덮어씀.

**준비물**: ① Drive 에 `family_v1.npz`(이미 있음) ② Drive 에 `mobilenet_v3_large_v2_best.pth` 업로드 (로컬 `E:\Terra\weights\`).  ▶ 런타임 GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# !ls '/content/drive/MyDrive'   # 파일명 확인용

In [ ]:
!pip install -q onnxscript onnx onnxruntime

In [ ]:
# ===== Config — v2 백본 =====
DATA_NPZ = '/content/drive/MyDrive/family_v1.npz'
BACKBONE = '/content/drive/MyDrive/mobilenet_v3_large_v2_best.pth'   # v2 (100-class)
OUT_ONNX = 'mobilenet_v3_family_v1_v2bb.onnx'                        # v3 onnx 안 덮어쓰게 다른 이름

NUM_CLASSES   = 5
EPOCHS        = 10
BATCH         = 64
LR            = 1e-3
LABEL_SMOOTH  = 0.05
OPSET         = 13
SEED          = 0

import os, numpy as np, torch
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
assert os.path.exists(DATA_NPZ), 'DATA_NPZ 경로 확인: ' + DATA_NPZ
assert os.path.exists(BACKBONE), 'BACKBONE 경로 확인: ' + BACKBONE
print('device:', DEVICE, '| torch', torch.__version__)
print('backbone:', BACKBONE)

In [ ]:
z = np.load(DATA_NPZ, allow_pickle=True)
X_train, y_train = z['X_train'], z['y_train'].astype(np.int64)
X_val,   y_val   = z['X_val'],   z['y_val'].astype(np.int64)
pid_map = dict(z['pid_map'])
CLASS_NAMES = [str(pid_map.get(i, 'cls%d' % i)) for i in range(NUM_CLASSES)]
if (NUM_CLASSES - 1) not in pid_map:
    CLASS_NAMES[NUM_CLASSES - 1] = 'unknown'

print('X_train', X_train.shape, X_train.dtype, '| X_val', X_val.shape)
print('classes:', {i: CLASS_NAMES[i] for i in range(NUM_CLASSES)})
for split, y in [('train', y_train), ('val', y_val)]:
    u, c = np.unique(y, return_counts=True)
    print(split, {CLASS_NAMES[int(k)]: int(v) for k, v in zip(u, c)})

In [ ]:
from torch.utils.data import Dataset, DataLoader

# jetson_infer.to_input 과 동일 (ImageNet mean/std, /255, CHW)
_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def preprocess(img_uint8_hwc):
    x = torch.from_numpy(np.ascontiguousarray(img_uint8_hwc)).float().permute(2, 0, 1) / 255.0
    return (x - _MEAN) / _STD

class FootstepDS(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return preprocess(self.X[i]), int(self.y[i])

train_loader = DataLoader(FootstepDS(X_train, y_train), batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(FootstepDS(X_val,   y_val),   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
print('train batches', len(train_loader), '| val batches', len(val_loader))

In [ ]:
import torch.nn as nn
from torchvision.models import mobilenet_v3_large

obj = torch.load(BACKBONE, map_location='cpu', weights_only=False)
print('ckpt keys:', list(obj.keys()))
print('num_classes:', obj.get('num_classes'))
print('val_acc:', obj.get('val_acc'))
print('epoch:', obj.get('epoch'))

sd = None
for key in ('model_state_dict', 'state_dict', 'model'):
    if key in obj and isinstance(obj[key], dict):
        sd = obj[key]
        break
if sd is None:
    sd = obj

def _strip(k):
    for p in ('module.', 'model.'):
        if k.startswith(p):
            k = k[len(p):]
    return k

sd = {_strip(k): v for k, v in sd.items()}

orig_nc = int(obj.get('num_classes') or sd['classifier.3.weight'].shape[0])
print('orig num_classes:', orig_nc)

model = mobilenet_v3_large(weights=None)
in_f = model.classifier[3].in_features
model.classifier[3] = nn.Linear(in_f, orig_nc)
missing, unexpected = model.load_state_dict(sd, strict=False)
print('missing:', len(missing))
print('unexpected:', len(unexpected))
assert len(missing) <= 2
assert len(unexpected) <= 2

for p in model.parameters():
    p.requires_grad = False
model.classifier[3] = nn.Linear(in_f, NUM_CLASSES)
model = model.to(DEVICE)
print('done. trainable:')
print([n for n, p in model.named_parameters() if p.requires_grad])

In [ ]:
u, c = np.unique(y_train, return_counts=True)
w = np.zeros(NUM_CLASSES, dtype=np.float32)
w[u] = (len(y_train) / (len(u) * c)).astype(np.float32)
class_w = torch.tensor(w, device=DEVICE)
print('class weights:', {CLASS_NAMES[i]: round(float(w[i]), 3) for i in range(NUM_CLASSES)})

criterion = nn.CrossEntropyLoss(weight=class_w, label_smoothing=LABEL_SMOOTH)
optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=LR)

In [ ]:
@torch.no_grad()
def evaluate(loader):
    model.eval()
    correct = total = 0
    cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    for xb, yb in loader:
        xb = xb.to(DEVICE)
        pred = model(xb).argmax(1).cpu().numpy()
        yb = yb.numpy()
        for t, p in zip(yb, pred):
            cm[t, p] += 1
        correct += int((pred == yb).sum()); total += len(yb)
    return correct / max(total, 1), cm

for epoch in range(1, EPOCHS + 1):
    model.train()
    run_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward(); optimizer.step()
        run_loss += loss.item() * len(yb)
    val_acc, _ = evaluate(val_loader)
    print('epoch %2d/%d  loss=%.4f  val_acc=%.4f' % (epoch, EPOCHS, run_loss / len(y_train), val_acc))

In [ ]:
import matplotlib.pyplot as plt
val_acc, cm = evaluate(val_loader)
print('final val accuracy: %.4f' % val_acc)
fam_correct = sum(cm[i, i] for i in range(NUM_CLASSES - 1))
fam_total = sum(cm[i].sum() for i in range(NUM_CLASSES - 1))
print('family-only acc: %.4f (%d/%d)' % (fam_correct / max(fam_total, 1), fam_correct, fam_total))
leak = sum(cm[i, NUM_CLASSES - 1] for i in range(NUM_CLASSES - 1))
print('family -> unknown 누수:', int(leak))
print('\nper-class recall:')
for i in range(NUM_CLASSES):
    tot = cm[i].sum()
    print('  %-10s %.3f  (%d/%d)' % (CLASS_NAMES[i], cm[i, i] / max(tot, 1), cm[i, i], tot))

fig, ax = plt.subplots(figsize=(5.5, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(['0','1','2','3','unk'])
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(['0','1','2','3','unk'])
ax.set_xlabel('pred'); ax.set_ylabel('true'); ax.set_title('v2bb val confusion (acc %.3f)' % val_acc)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, int(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > cm.max() * 0.5 else 'black')
fig.colorbar(im); plt.tight_layout(); plt.show()

In [ ]:
model.eval().to('cpu')
dummy = torch.randn(1, 3, 224, 224)
torch.onnx.export(
    model,
    dummy,
    OUT_ONNX,
    input_names=['input'],
    output_names=['logits'],
    opset_version=13,
    dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}},
    do_constant_folding=True,
    dynamo=False,
)
mb = os.path.getsize(OUT_ONNX) / 1e6
print('saved:', OUT_ONNX)
print('size MB:', round(mb, 1))
assert mb > 10

In [ ]:
import onnxruntime as ort
sess = ort.InferenceSession(OUT_ONNX, providers=['CPUExecutionProvider'])
with torch.no_grad():
    ref = model(dummy).numpy()
got = sess.run(['logits'], {'input': dummy.numpy()})[0]
print('max diff:', float(np.abs(ref - got).max()))

In [ ]:
from google.colab import files
files.download(OUT_ONNX)
# import shutil; shutil.copy(OUT_ONNX, '/content/drive/MyDrive/')   # Drive 로 복사하려면 주석 해제

## 비교 판단
v3 기준선: **val 86.0% / family-only 78.8% / family→unknown 0 / unknown recall 97%**.
- v2 의 **family-only acc** 가 더 높으면 → v2bb ONNX 배포.
- **family→unknown 누수**가 0 유지되는지 꼭 확인 (시연 안정성 직결).
- unknown recall 도 95%+ 유지 확인.

배포: `scp mobilenet_v3_family_v1_v2bb.onnx snup2@snup2-desktop:~/terra/` → `trtexec --fp16` → `web_server.py --stm32 --plan <plan>`.